# Stage 02: Data Integrity and Cleaning

This stage determines whether defensible changes are necessary before analysis. Cleaning choices can affect treatment-effect estimates, so observations are changed only when there is clear evidence that doing so is justified. The original raw dataset remains untouched.

Stage 01 identified 64,000 rows, 12 columns, no missing values, 6,562 exact duplicate rows, no explicit customer identifier, zero-inflated `visit`, `conversion`, and `spend` outcomes, and a long-tailed positive-spend distribution. This notebook verifies and investigates those observations; it does not perform experiment-effect inference, sample-ratio-mismatch testing, power analysis, or treatment recommendations.

## 1. Load and verify the raw data

All paths are project-relative. The raw file is read only.

In [1]:
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path('.')
if not (project_root / 'data/raw/Hillstrom.csv').exists():
    project_root = Path('..')

raw_path = project_root / 'data/raw/Hillstrom.csv'
processed_path = project_root / 'data/processed/Hillstrom_clean.csv'

def file_sha256(path):
    return sha256(path.read_bytes()).hexdigest()

raw_hash_before = file_sha256(raw_path)
raw_data = pd.read_csv(raw_path)

print(f'Raw dataset: {raw_path}')
print(f'Shape: {raw_data.shape[0]:,} rows × {raw_data.shape[1]} columns')
print('Columns:', raw_data.columns.tolist())
display(raw_data.dtypes.rename('data_type').to_frame())
print(f'Missing cells: {raw_data.isna().sum().sum():,}')
print(f'Exact duplicate rows: {raw_data.duplicated().sum():,}')

Raw dataset: ../data/raw/Hillstrom.csv
Shape: 64,000 rows × 12 columns
Columns: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']


,data_type
recency,int64
history_segment,str
history,float64
mens,int64
womens,int64
zip_code,str
newbie,int64
channel,str
segment,str
visit,int64


Missing cells: 0
Exact duplicate rows: 6,562


The observed shape, schema, absence of missing values, and duplicate count confirm the Stage 01 profile.

## 2. Investigate exact duplicate rows

An identical row is not sufficient evidence that the same experimental unit was recorded more than once: the file has no customer identifier, and distinct customers can share every observed characteristic and outcome. The following profile therefore describes duplicates without deleting them.

In [2]:
row_group_sizes = raw_data.value_counts(dropna=False)
duplicate_groups = row_group_sizes[row_group_sizes > 1]
duplicate_records = raw_data.loc[raw_data.duplicated(keep=False)].copy()

print(f'Unique rows: {len(row_group_sizes):,}')
print(f'Duplicated rows using pandas duplicated(): {raw_data.duplicated().sum():,}')
print(f'Duplicate groups: {len(duplicate_groups):,}')
print(f'Maximum identical-row group size: {duplicate_groups.max():,}')

duplicate_group_size_distribution = duplicate_groups.value_counts().sort_index().rename_axis('identical_row_count').reset_index(name='number_of_groups')
display(duplicate_group_size_distribution)

print('Frequently repeated rows (top five):')
display(duplicate_groups.head().rename('occurrences').reset_index())

duplicate_by_treatment = duplicate_records['segment'].value_counts().rename_axis('segment').reset_index(name='records_in_duplicate_groups')
duplicate_by_treatment['percentage_of_duplicate_records'] = (duplicate_by_treatment['records_in_duplicate_groups'] / len(duplicate_records) * 100).round(2)
print('Duplicate-group records by observed treatment label:')
display(duplicate_by_treatment)

duplicate_outcomes = duplicate_records.groupby(['visit', 'conversion', 'spend']).size().sort_values(ascending=False).head(10).rename('records').reset_index()
print('Most common outcome combinations among records in duplicate groups:')
display(duplicate_outcomes)
print(f'Records in duplicate groups with visit = conversion = spend = 0: {((duplicate_records.visit == 0) & (duplicate_records.conversion == 0) & (duplicate_records.spend == 0)).sum():,} of {len(duplicate_records):,}')

Unique rows: 57,438
Duplicated rows using pandas duplicated(): 6,562
Duplicate groups: 1,072
Maximum identical-row group size: 31


,identical_row_count,number_of_groups
0,2,224
1,3,121
2,4,92
3,5,77
4,6,64
5,7,69
6,8,76
7,9,61
8,10,53
9,11,46


Frequently repeated rows (top five):


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend,occurrences
0,10,1) $0 - $100,29.99,0,1,Surburban,1,Phone,Mens E-Mail,0,0,0.0,31
1,10,1) $0 - $100,29.99,1,0,Surburban,0,Phone,No E-Mail,0,0,0.0,27
2,10,1) $0 - $100,29.99,1,0,Surburban,0,Web,Womens E-Mail,0,0,0.0,27
3,10,1) $0 - $100,29.99,0,1,Urban,1,Phone,Womens E-Mail,0,0,0.0,27
4,10,1) $0 - $100,29.99,1,0,Surburban,1,Web,No E-Mail,0,0,0.0,26


Duplicate-group records by observed treatment label:


,segment,records_in_duplicate_groups,percentage_of_duplicate_records
0,Womens E-Mail,2575,33.73
1,No E-Mail,2555,33.47
2,Mens E-Mail,2504,32.80


Most common outcome combinations among records in duplicate groups:


,visit,conversion,spend,records
0,0,0,0.0,7012
1,1,0,0.0,622


Records in duplicate groups with visit = conversion = spend = 0: 7,012 of 7,634


**Decision:** retain all duplicate rows. Duplicate records appear in every treatment group and commonly combine ordinary customer characteristics with zero outcomes. Without an identifier, this evidence does not distinguish data duplication from distinct customers with the same observed profile; removing them would introduce an unsupported change to the experimental data.

## 3. Categorical-variable integrity

Categories are reported as observed. Whitespace and case checks look for formatting inconsistencies without changing labels; in particular, `Surburban` is retained exactly as supplied.

In [3]:
categorical_columns = ['history_segment', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion']
binary_columns = ['mens', 'womens', 'newbie', 'visit', 'conversion']

category_checks = []
for column in categorical_columns:
    values = raw_data[column].astype(str)
    counts = raw_data[column].value_counts(dropna=False).rename_axis(column).reset_index(name='count')
    counts['percentage'] = (counts['count'] / len(raw_data) * 100).round(2)
    print(f'\n{column}')
    display(counts)
    category_checks.append({
        'variable': column,
        'observed_categories': raw_data[column].nunique(dropna=False),
        'leading_or_trailing_whitespace': int(values.str.strip().ne(values).sum()),
        'case_only_collisions': int(values.str.lower().nunique() < values.nunique()),
        'invalid_binary_values': int((~raw_data[column].isin([0, 1])).sum()) if column in binary_columns else 'Not applicable',
    })

display(pd.DataFrame(category_checks))


history_segment


,history_segment,count,percentage
0,1) $0 - $100,22970,35.89
1,2) $100 - $200,14254,22.27
2,3) $200 - $350,12289,19.20
3,4) $350 - $500,6409,10.01
4,5) $500 - $750,4911,7.67
5,"6) $750 - $1,000",1859,2.90
6,"7) $1,000 +",1308,2.04



mens


,mens,count,percentage
0,1,35266,55.1
1,0,28734,44.9



womens


...


visit


,visit,count,percentage
0,0,54606,85.32
1,1,9394,14.68



conversion


,conversion,count,percentage
0,0,63422,99.1
1,1,578,0.9


,variable,observed_categories,leading_or_trailing_whitespace,case_only_collisions,invalid_binary_values
0,history_segment,7,0,0,Not applicable
1,mens,2,0,0,0
2,womens,2,0,0,0
3,zip_code,3,0,0,Not applicable
4,newbie,2,0,0,0
5,channel,3,0,0,Not applicable
6,segment,3,0,0,Not applicable
7,visit,2,0,0,0
8,conversion,2,0,0,0


## 4. Numeric-variable integrity

High values are documented rather than removed. A separate summary of positive spend makes the non-zero portion of this zero-inflated outcome visible.

In [4]:
numeric_columns = ['recency', 'history', 'spend']
numeric_summary = raw_data[numeric_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T.rename(columns={'50%': 'median'})
numeric_summary['zero_count'] = (raw_data[numeric_columns] == 0).sum()
numeric_summary['negative_count'] = (raw_data[numeric_columns] < 0).sum()
display(numeric_summary)

positive_spend = raw_data.loc[raw_data['spend'] > 0, 'spend']
positive_spend_summary = positive_spend.describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).rename(index={'50%': 'median'})
print(f'Positive-spend observations: {len(positive_spend):,}')
display(positive_spend_summary.to_frame('positive_spend'))

positive_q1, positive_q3 = positive_spend.quantile([0.25, 0.75])
positive_spend_iqr_upper = positive_q3 + 1.5 * (positive_q3 - positive_q1)
print(f'Positive-spend IQR screening threshold: {positive_spend_iqr_upper:.2f}')
print(f'Positive-spend observations above this screening threshold: {(positive_spend > positive_spend_iqr_upper).sum():,}')

,count,mean,std,min,1%,5%,25%,median,75%,95%,99%,max,zero_count,negative_count
recency,64000.0,5.763734,3.507592,1.00,1.00,1.00,2.00,6.00,9.0000,11.0000,12.0000,12.00,0,0
history,64000.0,242.085656,256.158608,29.99,29.99,29.99,64.66,158.11,325.6575,747.2625,1218.6643,3345.93,0,0
spend,64000.0,1.050908,15.036448,0.00,0.00,0.00,0.00,0.00,0.0000,0.0000,0.0000,499.00,63422,0


Positive-spend observations: 578


,positive_spend
count,578.000000
mean,116.363547
std,107.871537
min,29.990000
1%,29.990000
5%,29.990000
25%,32.272500
median,80.795000
75%,153.350000
95%,362.200000


Positive-spend IQR screening threshold: 334.97
Positive-spend observations above this screening threshold: 39


## 5. Logical consistency of outcomes

These checks identify potentially suspicious combinations but do not alter observations automatically.

In [5]:
outcome_checks = {
    'conversion = 1 and visit = 0': ((raw_data['conversion'] == 1) & (raw_data['visit'] == 0)).sum(),
    'spend > 0 and conversion = 0': ((raw_data['spend'] > 0) & (raw_data['conversion'] == 0)).sum(),
    'spend > 0 and visit = 0': ((raw_data['spend'] > 0) & (raw_data['visit'] == 0)).sum(),
    'conversion = 1 and spend = 0': ((raw_data['conversion'] == 1) & (raw_data['spend'] == 0)).sum(),
    'negative spend': (raw_data['spend'] < 0).sum(),
    'visit outside binary range': (~raw_data['visit'].isin([0, 1])).sum(),
    'conversion outside binary range': (~raw_data['conversion'].isin([0, 1])).sum(),
}
outcome_check_summary = pd.DataFrame(outcome_checks.items(), columns=['check', 'rows_affected'])
display(outcome_check_summary)
print('No correction is justified when these checks return zero; exact outcome definitions and observation windows remain undocumented.')

,check,rows_affected
0,conversion = 1 and visit = 0,0
1,spend > 0 and conversion = 0,0
2,spend > 0 and visit = 0,0
3,conversion = 1 and spend = 0,0
4,negative spend,0
5,visit outside binary range,0
6,conversion outside binary range,0


No correction is justified when these checks return zero; exact outcome definitions and observation windows remain undocumented.


## 6. Structural relationship checks

The labels in `history_segment` state monetary boundaries, allowing a direct consistency check against `history`. This verifies the observed relationship without rewriting either field.

In [6]:
history_segment_rules = [
    ('1) $0 - $100', 0, 100), ('2) $100 - $200', 100, 200),
    ('3) $200 - $350', 200, 350), ('4) $350 - $500', 350, 500),
    ('5) $500 - $750', 500, 750), ('6) $750 - $1,000', 750, 1000),
    ('7) $1,000 +', 1000, np.inf),
]
history_segment_profile = raw_data.groupby('history_segment', observed=True)['history'].agg(['count', 'min', 'max']).reset_index()
display(history_segment_profile)

expected_history_segment = np.select(
    [
        raw_data['history'].between(lower, upper, inclusive='left')
        for _, lower, upper in history_segment_rules
    ],
    [label for label, _, _ in history_segment_rules],
    default='unassigned',
)
history_segment_inconsistencies = (raw_data['history_segment'] != expected_history_segment).sum()
print(f'History/history_segment inconsistencies using the displayed band boundaries: {history_segment_inconsistencies:,}')

,history_segment,count,min,max
0,1) $0 - $100,22970,29.99,99.99
1,2) $100 - $200,14254,100.00,199.98
2,3) $200 - $350,12289,200.00,349.96
3,4) $350 - $500,6409,350.01,499.95
4,5) $500 - $750,4911,500.00,749.79
5,"6) $750 - $1,000",1859,750.01,999.75
6,"7) $1,000 +",1308,1000.15,3345.93


History/history_segment inconsistencies using the displayed band boundaries: 0


## 7. Data-type assessment

The imported types are retained in the saved CSV. Several integer fields are conceptually binary and several text fields are conceptually categorical or ordered categorical, but changing in-memory types would not improve the CSV values or provide a defensible cleaning action at this stage. Their conceptual roles are documented for later analysis.

In [7]:
type_assessment = pd.DataFrame([
    ['recency', str(raw_data.recency.dtype), 'numeric / ordered discrete', 'Retain', 'Observed 1–12 numeric scale.'],
    ['history_segment', str(raw_data.history_segment.dtype), 'ordered categorical', 'Retain', 'Text labels encode ordered history bands.'],
    ['history', str(raw_data.history.dtype), 'numeric', 'Retain', 'Continuous monetary-looking measure.'],
    ['mens, womens, newbie', 'int64', 'binary', 'Retain', 'All observed values are 0 or 1.'],
    ['zip_code, channel, segment', 'str', 'categorical', 'Retain', 'Observed labels are valid and unchanged.'],
    ['visit, conversion', 'int64', 'binary outcome', 'Retain', 'All observed values are 0 or 1.'],
    ['spend', str(raw_data.spend.dtype), 'numeric outcome', 'Retain', 'Non-negative values; preserve precision.'],
], columns=['variable(s)', 'original_type', 'conceptual_type', 'decision', 'reason'])
display(type_assessment)

,variable(s),original_type,conceptual_type,decision,reason
0,recency,int64,numeric / ordered discrete,Retain,Observed 1–12 numeric scale.
1,history_segment,str,ordered categorical,Retain,Text labels encode ordered history bands.
2,history,float64,numeric,Retain,Continuous monetary-looking measure.
3,"mens, womens, newbie",int64,binary,Retain,All observed values are 0 or 1.
4,"zip_code, channel, segment",str,categorical,Retain,Observed labels are valid and unchanged.
5,"visit, conversion",int64,binary outcome,Retain,All observed values are 0 or 1.
6,spend,float64,numeric outcome,Retain,Non-negative values; preserve precision.


## 8. Outlier policy

This project does not remove observations solely because they are statistically extreme. Legitimate high-spend customer behavior is distinct from impossible or invalid data. Therefore, high-spend observations are preserved and documented for potential later sensitivity analysis. `spend` is not winsorized or trimmed in Stage 02 because this review found no evidence that the observed high values are invalid.

## 9. Cleaning decision log

Every investigated issue is recorded below, including decisions not to make a change.

In [8]:
cleaning_decision_log = pd.DataFrame([
    ['Missing values', int(raw_data.isna().sum().sum()), 'Counted missing cells across every column.', 'No change', 'None', 'No missing values observed.'],
    ['Exact duplicate rows', int(raw_data.duplicated().sum()), 'Profiled 1,072 duplicate groups; no identifier is available.', 'Retain', 'None', 'Identical values do not establish duplicate experimental units.'],
    ['Invalid categories / formatting', 0, 'Checked categories, whitespace, case-only collisions, and binary values.', 'No change', 'None', 'No invalid binary values or formatting inconsistencies observed.'],
    ['Outcome inconsistencies', int(outcome_check_summary.rows_affected.sum()), 'Checked seven logical combinations and binary ranges.', 'No change', 'None', 'All specified checks returned zero.'],
    ['Negative values', int((raw_data[numeric_columns] < 0).sum().sum()), 'Checked recency, history, and spend.', 'No change', 'None', 'No negative values observed.'],
    ['Extreme spend', int((positive_spend > positive_spend_iqr_upper).sum()), 'Reviewed positive-spend distribution and IQR screening flags.', 'Retain', 'None', 'High spend alone is not evidence of invalid data.'],
    ['Data types', 0, 'Assessed imported and conceptual types.', 'No change', 'None', 'Existing types preserve values; conceptual roles are documented.'],
    ['History/history_segment consistency', int(history_segment_inconsistencies), 'Compared history to displayed segment boundaries.', 'No change', 'None', 'All rows match the displayed bands.'],
], columns=['Issue', 'Rows Affected', 'Investigation', 'Decision', 'Action Taken', 'Reason'])
display(cleaning_decision_log)

,Issue,Rows Affected,Investigation,Decision,Action Taken,Reason
0,Missing values,0,Counted missing cells across every column.,No change,None,No missing values observed.
1,Exact duplicate rows,6562,"Profiled 1,072 duplicate groups; no identifier...",Retain,None,Identical values do not establish duplicate ex...
2,Invalid categories / formatting,0,"Checked categories, whitespace, case-only coll...",No change,None,No invalid binary values or formatting inconsi...
3,Outcome inconsistencies,0,Checked seven logical combinations and binary ...,No change,None,All specified checks returned zero.
4,Negative values,0,"Checked recency, history, and spend.",No change,None,No negative values observed.
5,Extreme spend,39,Reviewed positive-spend distribution and IQR s...,Retain,None,High spend alone is not evidence of invalid data.
6,Data types,0,Assessed imported and conceptual types.,No change,None,Existing types preserve values; conceptual rol...
7,History/history_segment consistency,0,Compared history to displayed segment boundaries.,No change,None,All rows match the displayed bands.


## 10. Create the analysis-ready dataset

No row-level cleaning was justified. The processed file is therefore a value-preserving copy of the raw dataset, created to establish the approved analysis input without altering the raw source.

In [9]:
analysis_ready_data = raw_data.copy()
analysis_ready_data.to_csv(processed_path, index=False)
print(f'Created processed dataset: {processed_path}')
print(f'Processed shape: {analysis_ready_data.shape[0]:,} rows × {analysis_ready_data.shape[1]} columns')

Created processed dataset: ../data/processed/Hillstrom_clean.csv
Processed shape: 64,000 rows × 12 columns


## 11. Validate the output

The validation compares dimensions, treatment allocation, missingness, outcome counts, and total spend. It also confirms that the raw file did not change while this notebook ran.

In [10]:
processed_data = pd.read_csv(processed_path)
validation_summary = pd.DataFrame({
    'metric': ['rows', 'columns', 'missing_cells', 'visit_ones', 'conversion_ones', 'total_spend'],
    'raw': [len(raw_data), raw_data.shape[1], raw_data.isna().sum().sum(), (raw_data.visit == 1).sum(), (raw_data.conversion == 1).sum(), raw_data.spend.sum()],
    'processed': [len(processed_data), processed_data.shape[1], processed_data.isna().sum().sum(), (processed_data.visit == 1).sum(), (processed_data.conversion == 1).sum(), processed_data.spend.sum()],
})
validation_summary['difference'] = validation_summary['processed'] - validation_summary['raw']
display(validation_summary)

treatment_validation = pd.concat([
    raw_data['segment'].value_counts().rename('raw'),
    processed_data['segment'].value_counts().rename('processed'),
], axis=1)
treatment_validation['difference'] = treatment_validation['processed'] - treatment_validation['raw']
display(treatment_validation)

raw_hash_after = file_sha256(raw_path)
print(f'Raw file unchanged during notebook execution: {raw_hash_before == raw_hash_after}')
print(f'Processed values and imported data types match raw: {raw_data.equals(processed_data)}')

,metric,raw,processed,difference
0,rows,64000.00,64000.00,0.0
1,columns,12.00,12.00,0.0
2,missing_cells,0.00,0.00,0.0
3,visit_ones,9394.00,9394.00,0.0
4,conversion_ones,578.00,578.00,0.0
5,total_spend,67258.13,67258.13,0.0


,raw,processed,difference
segment,,,
Womens E-Mail,21387,21387,0
Mens E-Mail,21307,21307,0
No E-Mail,21306,21306,0


Raw file unchanged during notebook execution: True
Processed values and imported data types match raw: True


## Stage 02 Findings

All requested integrity checks were completed. The 6,562 exact duplicate rows were retained because there is no customer identifier and identical observed records are not proof of repeated experimental units. No missing values, invalid binary values, category-formatting issues, negative numeric values, outcome-logic conflicts, or `history`/`history_segment` inconsistencies were found.

High spend values were retained under the project outlier policy: the positive-spend distribution is long-tailed, but no evidence identifies these observations as invalid. No data types, rows, columns, labels, or values were changed. The processed dataset contains the same 64,000 rows and 12 columns as the raw input.

Remaining limitations include the lack of an explicit experimental-unit identifier, undocumented variable definitions and timing, duplicate-row ambiguity, and outcome zero inflation. These are validation and analysis considerations for later stages, not grounds for unsubstantiated cleaning.